In [1]:
from mces_splitting import split_dataset_lower_bound_only
import polars as pl
from hrms_utils.rdkit import inchi_to_smiles_polars

In [2]:
smrt = pl.read_csv("../data/SMRT_dataset.csv",separator=";")
print(smrt)

shape: (80_038, 3)
┌──────────┬───────┬─────────────────────────────────┐
│ pubchem  ┆ rt    ┆ inchi                           │
│ ---      ┆ ---   ┆ ---                             │
│ i64      ┆ f64   ┆ str                             │
╞══════════╪═══════╪═════════════════════════════════╡
│ 5139     ┆ 93.5  ┆ InChI=1S/C3H8N2S/c1-2-6-3(4)5/… │
│ 3505     ┆ 687.8 ┆ InChI=1S/C19H25Cl2N3O3/c1-27-1… │
│ 2159     ┆ 590.7 ┆ InChI=1S/C17H27N3O4S/c1-4-20-8… │
│ 1340     ┆ 583.6 ┆ InChI=1S/C9H7NO2/c11-8-3-1-2-7… │
│ 3344     ┆ 579.0 ┆ InChI=1S/C15H20N2O2/c18-14-16-… │
│ …        ┆ …     ┆ …                               │
│ 97733655 ┆ 946.4 ┆ InChI=1S/C25H29N3O6S/c1-5-24(2… │
│ 98666786 ┆ 653.1 ┆ InChI=1S/C17H24FN3O5S/c1-25-7-… │
│ 98670835 ┆ 648.2 ┆ InChI=1S/C17H25N3O5S/c1-13-4-3… │
│ 98779314 ┆ 783.9 ┆ InChI=1S/C21H25N3O4S/c1-15-7-9… │
│ 99905419 ┆ 603.7 ┆ InChI=1S/C14H16N2O2/c17-13-11-… │
└──────────┴───────┴─────────────────────────────────┘


In [4]:
smrt = smrt.with_columns(
    smiles=pl.col("inchi").map_batches(
        function=inchi_to_smiles_polars,
        return_dtype=pl.String,
        # is_elementwise=True
    )
)

In [ ]:
train_set, val_set, test_set = split_dataset_lower_bound_only(
    smrt.select(["smiles"]).to_series().to_list(),
    validation_fraction=0.1,
    test_fraction=0.1,
    mces_matrix_save_path="../data/mces_lower_bound_matrix.npy"
)